In [ ]:
## 1. Data Import and Validation

This section imports daily historical price data for XLK, XLF, and XLE 
and checks the datasets for missing values, duplicate dates, inconsistent data types, 
and basic price anomalies before calculating returns and risk metrics.

In [1]:
import pandas as pd 
import numpy as np

In [5]:
xlk=pd.read_csv(r'D:\cross-sector-etf-risk-analysis\data\xlk.csv')
xle=pd.read_csv(r'D:\cross-sector-etf-risk-analysis\data\xle.csv')
xlf=pd.read_csv(r'D:\cross-sector-etf-risk-analysis\data\xlf.csv')

In [ ]:
## Validation Structure
1. Structure
   ↓
shape
head
dtypes

2. Completeness
   ↓
missing values

3. Uniqueness
   ↓
duplicate rows
duplicate dates

4. Financial Logic
   ↓
price > 0
High >= OHLC
Low <= OHLC

5. Distribution
   ↓
describe()

In [17]:
def validate_data(df, name):
    
    print(f"===== {name} Data Validation =====")
    
    print("\nShape:")
    print(df.shape)
    
    print("\nFirst 5 rows:")
    print(df.head())
    
    print("\nData types:")
    print(df.dtypes)
    
    print("\nMissing values:")
    print(df.isnull().sum())
    
    print("\nDuplicate rows:")
    print(df.duplicated().sum())
    
    print("\nDuplicate dates:")
    print(df["Date"].duplicated().sum())
    
    print("\nNon-positive prices:")
    print((df[["Open", "High", "Low", "Close"]] <= 0).sum())
    
    invalid_high = df[
        (df["High"] < df["Open"]) |
        (df["High"] < df["Close"]) |
        (df["High"] < df["Low"])
    ]
    
    invalid_low = df[
        (df["Low"] > df["Open"]) |
        (df["Low"] > df["Close"]) |
        (df["Low"] > df["High"])
    ]
    
    print("\nInvalid High rows:")
    print(len(invalid_high))
    
    print("\nInvalid Low rows:")
    print(len(invalid_low))
    
    print("\nSummary statistics:")
    print(df.describe())
    

In [18]:
validate_data(xlf, "XLF")

===== XLF Data Validation =====

Shape:
(898, 6)

First 5 rows:
         Date   Open   High    Low  Close    Volume
0  2023-01-12  35.90  36.06  35.62  35.85  56642385
1  2023-01-13  35.39  36.17  35.26  36.12  65920861
2  2023-01-17  36.03  36.05  35.77  35.88  65345169
3  2023-01-18  35.68  35.90  35.18  35.21  53693044
4  2023-01-19  34.78  34.98  34.56  34.80  61169520

Data types:
Date          str
Open      float64
High      float64
Low       float64
Close     float64
Volume      int64
dtype: object

Missing values:
Date      0
Open      0
High      0
Low       0
Close     0
Volume    0
dtype: int64

Duplicate rows:
0

Duplicate dates:
0

Non-positive prices:
Open     0
High     0
Low      0
Close    0
dtype: int64

Invalid High rows:
0

Invalid Low rows:
0

Summary statistics:
             Open        High         Low       Close        Volume
count  898.000000  898.000000  898.000000  898.000000  8.980000e+02
mean    44.691122   44.979790   44.420511   44.711459  4.308964e+07
s

In [19]:
validate_data(xle, "XLE")

===== XLE Data Validation =====

Shape:
(898, 6)

First 5 rows:
         Date     Open     High      Low    Close    Volume
0  2023-01-12  40.0537  40.8352  39.9769  40.5731  40640588
1  2023-01-13  40.5416  40.7292  40.1124  40.6320  35208254
2  2023-01-17  40.7268  41.1425  40.5507  40.7177  37560259
3  2023-01-18  40.9075  41.3637  39.9318  39.9723  42145095
4  2023-01-19  39.7420  40.6320  39.6562  40.4694  37959321

Data types:
Date          str
Open      float64
High      float64
Low       float64
Close     float64
Volume      int64
dtype: object

Missing values:
Date      0
Open      0
High      0
Low       0
Close     0
Volume    0
dtype: int64

Duplicate rows:
0

Duplicate dates:
0

Non-positive prices:
Open     0
High     0
Low      0
Close    0
dtype: int64

Invalid High rows:
0

Invalid Low rows:
0

Summary statistics:
             Open        High         Low       Close        Volume
count  898.000000  898.000000  898.000000  898.000000  8.980000e+02
mean    43.970998   4

In [20]:
validate_data(xlk,'XLK')

===== XLK Data Validation =====

Shape:
(898, 6)

First 5 rows:
        Date     Open     High      Low    Close    Volume
0 2023-01-12  63.2936  63.9536  62.5207  63.6655  15695120
1 2023-01-13  63.0639  63.9386  62.9166  63.8467  10375581
2 2023-01-17  63.7579  64.5117  63.6800  64.1358  10654106
3 2023-01-18  64.4972  64.8397  63.2691  63.3075  14241205
4 2023-01-19  62.8138  63.1663  62.3395  62.5692  14965843

Data types:
Date      datetime64[us]
Open             float64
High             float64
Low              float64
Close            float64
Volume             int64
dtype: object

Missing values:
Date      0
Open      0
High      0
Low       0
Close     0
Volume    0
dtype: int64

Duplicate rows:
0

Duplicate dates:
0

Non-positive prices:
Open     0
High     0
Low      0
Close    0
dtype: int64

Invalid High rows:
0

Invalid Low rows:
0

Summary statistics:
                             Date        Open        High         Low  \
count                         898  898.000000  8

In [ ]:
# # 2. Clean the dataset 
converting dates, sorting chronologically,removing duplicate dates, and resetting the index.

In [27]:
def clean_data(df):
    df=df.copy()
    
    df['Date']=pd.to_datetime(df['Date'])

    df=df.sort_values('Date')

    df=df.drop_duplicates(subset='Date')

    df-df.reset_index(drop=True)

    return df

    

In [28]:
xlk_clean=clean_data(xlk)

In [29]:
xle_clean=clean_data(xle)

In [31]:
xlf_clean=clean_data(xlf)

In [34]:
print(xlk_clean.info())
print(xlk_clean.head())
print(xlk_clean.tail())

<class 'pandas.DataFrame'>
RangeIndex: 898 entries, 0 to 897
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    898 non-null    datetime64[us]
 1   Open    898 non-null    float64       
 2   High    898 non-null    float64       
 3   Low     898 non-null    float64       
 4   Close   898 non-null    float64       
 5   Volume  898 non-null    int64         
dtypes: datetime64[us](1), float64(4), int64(1)
memory usage: 42.2 KB
None
        Date     Open     High      Low    Close    Volume
0 2023-01-12  63.2936  63.9536  62.5207  63.6655  15695120
1 2023-01-13  63.0639  63.9386  62.9166  63.8467  10375581
2 2023-01-17  63.7579  64.5117  63.6800  64.1358  10654106
3 2023-01-18  64.4972  64.8397  63.2691  63.3075  14241205
4 2023-01-19  62.8138  63.1663  62.3395  62.5692  14965843
          Date    Open    High      Low   Close   Volume
893 2026-08-06  183.19  186.91  182.060  185.33  6706016
894 2026-08-

In [35]:
print("XLK:", xlk_clean.shape)
print("XLF:", xlf_clean.shape)
print("XLE:", xle_clean.shape)

XLK: (898, 6)
XLF: (898, 6)
XLE: (898, 6)


In [36]:
print("XLK:", xlk_clean["Date"].min(), xlk_clean["Date"].max())
print("XLF:", xlf_clean["Date"].min(), xlf_clean["Date"].max())
print("XLE:", xle_clean["Date"].min(), xle_clean["Date"].max())

XLK: 2023-01-12 00:00:00 2026-08-12 00:00:00
XLF: 2023-01-12 00:00:00 2026-08-12 00:00:00
XLE: 2023-01-12 00:00:00 2026-08-12 00:00:00


In [37]:
print(xlk_clean['Date'].equals(xlf_clean['Date']))
print(xlf_clean['Date'].equals(xle_clean['Date']))

True
True
